# White Matter Synonym Integration
Classifies 312 input synonyms, maps each to the correct master tract row,
flags ambiguous/miscellaneous terms to `misc.csv`, and merges valid synonyms
into the `synonyms` column of `8April_master_v4.tsv`.

In [20]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────
MASTER_PATH = Path('./csvOutput/8April_master_v4.tsv')
OUT_MASTER  = Path('./csvOutput/8April_master_v5.csv')
OUT_MISC    = Path('./csvOutput/misc.csv')

df = pd.read_csv(MASTER_PATH, sep='\t', dtype=str).fillna('')
print(f'Master loaded: {len(df)} rows × {len(df.columns)} cols')
df[['clean_label', 'acronym', 'synonyms', 'source_scheme (toolbox)']].head(8)

Master loaded: 77 rows × 18 cols


,clean_label,acronym,synonyms,source_scheme (toolbox)
0,acoustic radiation,AR,auditory radiation,TRACULA ; PyAFQ
1,anterior commissure,AC ; ACOMM ; CA,anterior cerebral commissure ; commissura anterior cerebri ; commissura rostralis ; commissura anterior ; commissura...,TRACULA ; TractSeg
2,anterior thalamic radiation,ATR,corticospinal tract ; pyramidal tract ; tractus corticospinalis ; fibrae corticospinales,PyAFQ ; TractSeg ; TRACULA
3,arcuate fasciculus,AF ; AR,fasciculus arcuatus ; cerebral arcuate fasciculus,PyAFQ ; TractSeg ; TRACULA
4,callosum forceps major,FMA ; FMAJ,forceps major ; occipital forceps ; posterior forceps ; forceps major of corpus callosum,PyAFQ
5,callosum forceps minor,FMI ; FMIJ,forceps minor ; frontal forceps ; anterior forceps ; forceps minor of corpus callosum,PyAFQ
6,cingulum,CCG ; CG,cerebral crus ; pedunculus cerebri ; crus cerebri ; cerebral peduncle ; Cingulum (body structure),TractSeg
7,cingulum bundle dorsal,CBD,cingulum ; cingulum bundle ; cingulum of telencephalon ; neuraxis cingulum,TRACULA


## 1  Input synonym list

In [8]:
RAW_SYNONYMS = [
    "(Anterior) Cingulum Bundle", "Anteriofrontal Corpus Callosum", "Anterior Cerebral Commissure",
    "Anterior Commissural Nucleus", "Anterior Commissure", "Anterior Corpus Callosum",
    "Anterior Forceps", "Anterior Forceps Of Corpus Callosum", "Anterior Forceps Of The Corpus Callosum",
    "Anterior Radiation Of Thalamus", "Anterior Thalamic Radiation", "Anterior Thalamic Radiations",
    "Arcuate Fascicle", "Arcuate Fasciculus", "Asciculus, Perforating", "Aslant Tract",
    "Band Of Baillarger", "Basis Pedunculi", "Brain External Capsule", "Brain Fornix",
    "Brain Internal Capsule", "Capsula Externa", "Capsula Extrema", "Capsula Interna",
    "Cc - Corpus Callosum", "Cerebal Peduncle", "Cerebellar Peduncle", "Cerebellar Peduncle Structure",
    "Cerebellospinal Tract", "Cerebellum Peduncle", "Cerebral Arcuate Fasciculus", "Cerebral Fornix",
    "Cerebral Fornix Structure", "Cerebral Peduncle", "Cerebral Peduncle (Archaic)", "Cerebral Peduncle Structure",
    "Cerebral Uncinate Fasciculus", "Cingulate Cingulum", "Cingulum", "Cingulum (Ammon'S Horn)",
    "Cingulum (Hippocampus)", "Cingulum Bundle", "Cingulum Bundle In Hippocampus", "Cingulum Of Brain",
    "Cingulum Of Telencephalon", "Commissura Anterior", "Commissura Anterior Cerebri",
    "Commissura Rostral", "Commissura Rostralis", "Corona Radiata", "Corona Radiata Of Neuraxis",
    "Corpus Callosum", "Corpus Callosum - Anterior Midbody", "Corpus Callosum - Genu",
    "Corpus Callosum - Isthmus", "Corpus Callosum - Posterior Midbody", "Corpus Callosum - Rostral Body",
    "Corpus Callosum - Rostrum", "Corpus Callosum - Splenium", "Corpus Callosum External Capsule",
    "Corpus Callosum Structure", "Corpus Callosum, Anterior Forceps",
    "Corpus Callosum, Anterior Forceps (Arnold)", "Corpus Callosum, Forceps Major",
    "Corpus Callosum, Forceps Minor", "Corpus Callosum, Posterior Forceps (Arnold)",
    "Cortico-Pontine Fibers", "Cortico-Pontine Fibers, Pontine Part", "Corticopontine",
    "Corticopontine Fibers", "Corticopontine Fibers Of Pons", "Corticopontine Fibers Set",
    "Corticopontine Fibres", "Corticopontine Tract", "Corticopontine Tract Of Pons",
    "Corticospinal Fibers", "Corticospinal Tract", "Crus Cerebri", "External Capsule",
    "External Capsule Of Telencephalon", "External Sagittal Stratum", "Extreme Capsule",
    "Fasciculus Arcuatus", "Fasciculus Cerebro-Spinalis", "Fasciculus Fastigio-Vestibularis",
    "Fasciculus Fronto-Occipitalis Inferior", "Fasciculus Longitudinalis Inferior",
    "Fasciculus Longitudinalis Medialis (Pontis)", "Fasciculus Occipito-Frontalis Inferior",
    "Fasciculus Occipitofrontalis Inferior", "Fasciculus Occipitofrontalis Superior",
    "Fasciculus Pyramidalis", "Fasciculus Subcallosus", "Fastigiobulbar Tract",
    "Fibrae Arcuatae Cerebri", "Fibrae Corticopontinae", "Fibrae Corticospinales",
    "Fibrae Pontocerebellaris", "Forceps", "Forceps Frontalis", "Forceps Major",
    "Forceps Major Corporis Callosi", "Forceps Major Of Corpus Callosum",
    "Forceps Major Of The Corpus Callosum", "Forceps Minor", "Forceps Minor Corporis Callosi",
    "Forceps Minor Of Corpus Callosum", "Forceps Minor Of The Corpus Callosum",
    "Forceps Occipitalis", "Forebrain Fornix", "Fornix", "Fornix (Column And Body Of Fornix)",
    "Fornix Cerebri", "Fornix Hippocampus", "Fornix Of Brain", "Fornix Of Neuraxis",
    "Frontal Aslant Tract", "Frontal Forceps", "Fronto-Pontine Tract", "Fronto-Ponto-Cerebellar",
    "Fronto-Thalamic", "Frontotemporal Fasciculus", "Genicula-Celcarine Tract",
    "Geniculo-Calcarine Tract", "Geniculocalcarine Tract", "Geniculostriate Pathway",
    "Global", "Gratiolet'S Radiation", "Hippocampal Cingulum", "Hippocampus Cortex Cingulum",
    "Hippocampus Fornix", "Hook Bundle Of Russell", "Ic - Internal Capsule",
    "Inferior Cerebellar Peduncle", "Inferior Fronto-Occipital Fasciculus",
    "Inferior Longitudinal Fasciculus", "Inferior Occipitofrontal Fasciculus",
    "Internal Capsule", "Internal Capsule Of Brain", "Internal Capsule Of Telencephalon",
    "Internal Capsule Radiations", "Internal Capsule Structure", "Internal Capsule Structure Of Brain",
    "Lemniscus Medialis", "Major Forceps", "Mdlfang", "Mdlfspl", "Medial Lemniscus",
    "Medial Longitudinal Fasciculus", "Medial Longitudinal Fasciculus Of Pons",
    "Medial Longitudinal Fasciculus Of Pons Of Varolius",
    "Medial Longitudinal Fasciculus Of The Pons", "Medial Longitudinal Fasciculus Structure",
    "Meyer", "Meyer'S Loop", "Middle Cerebellar Peduncle", "Middle Frontal Corpus Callosum",
    "Middle Longitudinal Fasciculus",
    "Middle Longitudinal Fasciculus Connection To The Angular Gyrus",
    "Middle Longitudinal Fasciculus Connection To The Superior Parietal Lobe",
    "Minor Forceps", "Mlf-Medial Longitudinal Fasciculus", "Motor Cerebellar", "Motor Thalamic",
    "Na", "Neuraxis Cingulum", "Neuraxis Fornix", "Occipital Forceps",
    "Occipital Radiation Of Corpus Callosum", "Occipitocerebellar", "Olfactory Peduncle",
    "Olfactory Stalk", "Olfactory Tract", "Olfactory Tract Structure", "Optic Radiation",
    "Optic Radiations", "Paleocortical Commissure", "Parahippocampal Cingulum",
    "Parietal Corpus Callosum", "Parietal Radiation Of Corpus Callosum", "Parieto Thalamic",
    "Parieto-Occipital Pontine", "Parietocerebellar", "Path, Perforant", "Paths, Perforant",
    "Pathway, Perforant", "Pathways, Perforant", "Peduncle Of Midbrain",
    "Pedunclulus Olfactorius", "Pedunculi Cerebri", "Pedunculus Cerebralis",
    "Pedunculus Cerebri", "Perforant Path", "Perforant Paths", "Perforant Pathway",
    "Perforant Pathways", "Perforating Fasciculus", "Perpendicular Fasciculus",
    "Pons Medial Longitudinal Fasciculus", "Pons Of Varolius Medial Longitudinal Fasciculus",
    "Pontine Crossing Tract", "Pontocerebellar Fibers", "Pontocerebellar Tract",
    "Posteior Arcuate Fascisculus", "Posterior Arcuate Fasciculus", "Posterior Forceps",
    "Posterior Forceps Of Corpus Callosum", "Posterior Forceps Of The Corpus Callosum",
    "Precommisure", "Pyramid (Willis)", "Pyramidal Tract", "Radiatio Optica",
    "Radiatio Thalami Anterior", "Radiation Of Thalamus", "Radiationes Thalamicae Anteriores",
    "Railroad Nystagmus", "Reil'S Band", "Reil'S Ribbon", "Rostral Commissure",
    "Russell'S Fasciculus", "Sagittal Stratum", "Spinothalamic Tract",
    "Striato-Fronto-Orbital", "Striato-Occipital", "Striato-Parietal", "Striato-Postcentral",
    "Striato-Precentral", "Striato-Prefrontal", "Striato-Premotor",
    "Structure Of Anterior Commissure", "Structure Of Cerebral Cingulum",
    "Structure Of Cingulum", "Structure Of Corticopontine Tract Of Pons",
    "Structure Of Corticospinal Tract", "Structure Of External Capsule",
    "Structure Of Extreme Capsule", "Structure Of Forceps Major", "Structure Of Forceps Minor",
    "Structure Of Inferior Fronto-Occipital Fasciculus",
    "Structure Of Inferior Longitudinal Fasciculus", "Structure Of Optic Radiation",
    "Structure Of Superior Fronto-Occipital Fasciculus",
    "Structure Of Superior Longitudinal Fasciculus", "Structure Of Tapetum Of Corpus Callosum",
    "Structure Of Uncinate Fasciculus", "Structure Of Vertical Occipital Fasciculus",
    "Subcallosal Bundle", "Subcallosal Fasciculus", "Superior Cerebellar Peduncle",
    "Superior Fronto-Occipital Bundle", "Superior Fronto-Occipital Fasciculus",
    "Superior Longitudinal Fascicle", "Superior Longitudinal Fascicle I",
    "Superior Longitudinal Fascicle Ii", "Superior Longitudinal Fascicle Iii",
    "Superior Longitudinal Fasciculus", "Superior Occipito-Frontal Fascicle",
    "Superior Occipitofrontal Fasciculus", "Superior Thalamic Radiation", "Tapetum",
    "Tapetum Corporis Callosi", "Tapetum Of Corpus Callosum", "Tegmentum",
    "Temporo Thalamic", "Temporo-Parietal Connections To The Superior Parietal Lobule",
    "Temporo-Ponto-Cerebellar", "Temporooccipital Fasciculus", "Temporopulvinar",
    "Thalamic Radiation", "Thalamo-Postcentral", "Thalamo-Precentral",
    "Thalamo-Prefrontal", "Thalamo-Premotor", "Thalamus Radiation",
    "Tractus Cerebello-Bulbaris", "Tractus Cortico-Spinalis", "Tractus Corticopontinus",
    "Tractus Corticospinalis", "Tractus Olfactorium", "Tractus Olfactorius",
    "Tractus Pontocerebellaris", "Tractus Pyramidalis", "Tractus Uncinatus",
    "Tractus Uncinatus (Lewandowsky)", "Uncinate Bundle Of Russell",
    "Uncinate Fascicle (Russell)", "Uncinate Fasciculus", "Uncinate Fasciculus Of Cerebellum",
    "Uncinate Fasciculus Of Pons", "Uncinate Fasciculus Of Russell",
    "Uncinate Fasciculus Of The Pons", "Uncinate Fasciculus-2", "Vertical Occipital Fasciculus",
    "posterior cingulate", "stria terminalis", "retrolenticular part of internal capsule",
    "body of corpus callosum", "corpus callosum body", "truncus corporis callosi",
    "body of corpus callosum", "body of the corpus callosum", "corpus callosum truncus",
    "corpus callosum, corpus", "trunculus corporis callosi", "truncus corpus callosi",
    "trunk of corpus callosum", "posterior thalamic radiation", "posterior thalamic radiation",
    "posterior limb of internal capsule", "dentatothalamic tract", "dentatothalamic fibers",
    "tractus dentatothalamicus"
]

# Normalise: strip, title-case for display, lowercase for matching
def normalise(s):
    s = s.strip()
    # Remove leading/trailing punctuation artefacts
    s = re.sub(r'\s+', ' ', s)
    return s

raw = pd.DataFrame({'raw': RAW_SYNONYMS})
raw['clean'] = raw['raw'].apply(normalise)
raw = raw.drop_duplicates(subset='clean', keep='first').reset_index(drop=True)
print(f'Input synonyms (after dedup): {len(raw)}')

Input synonyms (after dedup): 310


## 2  Classification rules

Each synonym is assigned one of four categories:

| Category | Meaning | Destination |
|---|---|---|
| `mapped` | Clear 1-to-1 match to a master tract row | Added to `synonyms` column of that row |
| `new_tract` | Valid tract name not yet in master | New row added, flagged in `to_merge?` |
| `ambiguous` | Could match multiple tracts, or is a region/category label | `misc.csv` |
| `discard` | Noise, abbreviation stubs, duplicates of `clean_label` | `misc.csv` |

In [9]:
# ── Master lookup: clean_label → row index ────────────────────────────────
label_to_idx = {row['clean_label'].strip().lower(): i
                for i, row in df.iterrows()}

# ── Explicit classification map ───────────────────────────────────────────
# Each entry: normalised-lowercase synonym → (category, target_clean_label | note)
CLASSIFICATION = {
    # ── DISCARD: noise, stub abbreviations, uninformative ─────────────────
    'global':                           ('discard', 'Non-specific category label'),
    'na':                               ('discard', 'Placeholder / missing value'),
    'meyer':                            ('discard', 'Author name fragment; see meyer\'s loop'),
    'motor cerebellar':                 ('discard', 'Functional category, not a tract name'),
    'motor thalamic':                   ('discard', 'Functional category, not a tract name'),
    'parieto thalamic':                 ('discard', 'Functional category label'),
    'temporo thalamic':                 ('discard', 'Functional category label'),
    'fronto-thalamic':                  ('discard', 'Functional category label'),
    'fronto-ponto-cerebellar':          ('discard', 'Pathway category, not a single tract'),
    'temporo-ponto-cerebellar':         ('discard', 'Pathway category, not a single tract'),
    'parietocerebellar':                ('discard', 'Pathway category'),
    'occipitocerebellar':               ('discard', 'Pathway category'),
    'temporopulvinar':                  ('discard', 'Pathway category'),
    'mdlfang':                          ('discard', 'Sub-segment code (MdLF-Ang); add to master if needed'),
    'mdlfspl':                          ('discard', 'Sub-segment code (MdLF-Spl); add to master if needed'),
    'mlf-medial longitudinal fasciculus': ('discard', 'Redundant compound label — MLF already captured'),
    'railroad nystagmus':               ('discard', 'Clinical symptom, not a tract name'),
    'reil\'s band':                     ('discard', 'Archaic eponym for medial lemniscus — ambiguous'),
    'reil\'s ribbon':                   ('discard', 'Archaic eponym for medial lemniscus — ambiguous'),
    'forceps':                          ('discard', 'Generic; covered by forceps major / minor entries'),
    'corpus callosum, corpus':          ('discard', 'Malformed entry'),
    'tegmentum':                        ('discard', 'Brainstem region, not a white matter tract'),
    'posterior cingulate':              ('discard', 'Cortical region label, not a tract'),
    'cc - corpus callosum':             ('discard', 'Acronym expansion of CC — redundant'),
    'ic - internal capsule':            ('discard', 'Acronym expansion — redundant'),
    'cerebral peduncle structure':      ('discard', 'SNOMED structural label — covered by incoming relations'),
    'cerebellar peduncle structure':    ('discard', 'SNOMED structural label'),
    'cerebral fornix structure':        ('discard', 'SNOMED structural label'),
    'corpus callosum structure':        ('discard', 'SNOMED structural label'),
    'olfactory tract structure':        ('discard', 'SNOMED structural label'),
    'medial longitudinal fasciculus structure': ('discard', 'SNOMED structural label'),
    'corticopontine fibers set':        ('discard', 'SNOMED structural label variant'),
    'corona radiata of neuraxis':       ('discard', 'SNOMED structural label variant'),

    # ── AMBIGUOUS: maps to multiple rows or is a superstructure ───────────
    'corona radiata':                   ('ambiguous', 'Superstructure — contains ATR, STR, CST, PTR; maps to multiple rows'),
    'corpus callosum':                  ('ambiguous', 'Whole-structure label — applies to all CC_1–CC_7 rows'),
    'anterior corpus callosum':         ('ambiguous', 'Covers CC_1 rostrum through CC_4 anterior midbody'),
    'anteriofrontal corpus callosum':   ('ambiguous', 'Variant of anterior CC — overlaps CC_1/CC_2/CC_3'),
    'parietal corpus callosum':         ('ambiguous', 'Overlaps CC_5/CC_6; not precisely one subdivision'),
    'middle frontal corpus callosum':   ('ambiguous', 'Overlaps CC_3/CC_4; ambiguous subdivision mapping'),
    'internal capsule':                 ('ambiguous', 'Superstructure; contains ALIC, PLIC, retrolenticular parts'),
    'brain internal capsule':           ('ambiguous', 'Superstructure variant'),
    'internal capsule of brain':        ('ambiguous', 'Superstructure variant'),
    'internal capsule of telencephalon':('ambiguous', 'Superstructure variant'),
    'internal capsule radiations':      ('ambiguous', 'Superstructure — all thalamo-cortical radiations'),
    'internal capsule structure':       ('ambiguous', 'Superstructure SNOMED label'),
    'internal capsule structure of brain': ('ambiguous', 'Superstructure SNOMED label'),
    'sagittal stratum':                 ('ambiguous', 'Contains ILF + IFO; covers multiple tracts'),
    'cingulum':                         ('ambiguous', 'Applies to cingulum, cingulum bundle dorsal, cingulum bundle ventral, cingulum cingulate, cingulum hippocampus'),
    'cingulum bundle':                  ('ambiguous', 'Covers both CBD and CBV'),
    'cingulate cingulum':               ('ambiguous', 'Could be cingulum cingulate or cingulum — verify'),
    'cingulum of brain':                ('ambiguous', 'Whole cingulum system'),
    'hippocampal cingulum':             ('ambiguous', 'Overlaps cingulum hippocampus and cingulum bundle ventral'),
    'hippocampus cortex cingulum':      ('ambiguous', 'Overlaps cingulum hippocampus and cingulum bundle ventral'),
    'parahippocampal cingulum':         ('ambiguous', 'Overlaps cingulum hippocampus and cingulum bundle ventral'),
    'cerebellar peduncle':              ('ambiguous', 'Covers ICP, MCP, SCP — needs laterality/type specification'),
    'cerebellum peduncle':              ('ambiguous', 'Generic cerebellar peduncle'),
    'medial longitudinal fasciculus':   ('ambiguous', 'Could be middle longitudinal fasciculus (MdLF) OR the brainstem MLF — different tracts'),
    'medial longitudinal fasciculus of pons': ('ambiguous', 'Brainstem MLF; not same as MdLF'),
    'medial longitudinal fasciculus of pons of varolius': ('ambiguous', 'Brainstem MLF variant'),
    'medial longitudinal fasciculus of the pons': ('ambiguous', 'Brainstem MLF variant'),
    'pons medial longitudinal fasciculus': ('ambiguous', 'Brainstem MLF variant'),
    'pons of varolius medial longitudinal fasciculus': ('ambiguous', 'Brainstem MLF variant'),
    'fasciculus longitudinalis medialis (pontis)': ('ambiguous', 'Brainstem MLF (pontine); not MdLF'),
    'tapetum':                          ('ambiguous', 'Part of CC — maps to callosal rows but not a single subdivision'),
    'tapetum of corpus callosum':       ('ambiguous', 'Part of posterior CC; not a standalone row'),
    'tapetum corporis callosi':         ('ambiguous', 'Latin variant of tapetum of CC'),
    'cerebral peduncle':                ('ambiguous', 'Brainstem structure containing corticospinal + fronto-pontine + other fibers'),
    'basis pedunculi':                  ('ambiguous', 'Ventral cerebral peduncle; overlaps FPT and CST'),
    'cerebal peduncle':                 ('ambiguous', 'Typo variant of cerebral peduncle'),
    'cerebral peduncle (archaic)':      ('ambiguous', 'Archaic form of cerebral peduncle'),
    'crus cerebri':                     ('ambiguous', 'Synonym for cerebral peduncle — ambiguous'),
    'fasciculus fastigio-vestibularis': ('ambiguous', 'Fastigio-vestibular fibers; not a named tract in master'),
    'fastigiobulbar tract':             ('ambiguous', 'Cerebellar efferent; related to but distinct from SCP'),
    'cerebellospinal tract':            ('ambiguous', 'Broad cerebellar output; not a specific row'),
    'pontine crossing tract':           ('ambiguous', 'Pontocerebellar crossing fibers; related to MCP but distinct'),
    'corpus callosum external capsule': ('ambiguous', 'Malformed compound label — CC and external capsule are different structures'),
    'occipital radiation of corpus callosum': ('ambiguous', 'Callosal fibers to occipital lobe — maps to CC_7 or forceps major'),
    'parietal radiation of corpus callosum': ('ambiguous', 'Callosal fibers to parietal lobe — maps to CC_5 or CC_6'),
    'radiation of thalamus':            ('ambiguous', 'Generic thalamic radiation — covers ATR, STR, PTR, and others'),
    'thalamic radiation':               ('ambiguous', 'Generic thalamic radiation'),
    'thalamus radiation':               ('ambiguous', 'Generic thalamic radiation'),
    'posterior thalamic radiation':     ('ambiguous', 'Not a current master row; overlaps STR/thalamo-parietal/thalamo-occipital'),
    'perforant path':                   ('ambiguous', 'Hippocampal entorhinal-to-dentate pathway; not in master'),
    'perforant paths':                  ('ambiguous', 'Plural form of perforant path'),
    'perforant pathway':                ('ambiguous', 'Hippocampal pathway; not in master'),
    'perforant pathways':               ('ambiguous', 'Plural form'),
    'path, perforant':                  ('ambiguous', 'Inverted form of perforant path'),
    'paths, perforant':                 ('ambiguous', 'Inverted plural form'),
    'pathway, perforant':               ('ambiguous', 'Inverted form'),
    'pathways, perforant':              ('ambiguous', 'Inverted plural form'),
    'perforating fasciculus':           ('ambiguous', 'Old term for perforant path; hippocampal, not in master'),
    'asciculus, perforating':           ('ambiguous', 'Typo/inverted form of perforating fasciculus'),
    'perpendicular fasciculus':         ('ambiguous', 'Alternative name for vertical occipital fasciculus — verify before mapping'),
    'stria terminalis':                 ('ambiguous', 'Limbic tract; not in master — add as new_tract if needed'),
    'retrolenticular part of internal capsule': ('ambiguous', 'IC subdivision; not a standalone master row'),
    'posterior limb of internal capsule': ('ambiguous', 'IC subdivision containing CST/optic radiation; not a standalone row'),
    'aslant tract':                     ('ambiguous', 'Short form — could be frontal aslant tract or other'),
    'spinothalamic tract':              ('ambiguous', 'Sensory pathway; not in master — add as new_tract if needed'),
    'subcallosal fasciculus':           ('ambiguous', 'Synonym for superior fronto-occipital fasciculus OR a distinct bundle near CC — verify'),
    'fibrae pontocerebellaris':         ('ambiguous', 'Pontocerebellar fibers; related to MCP but overlaps'),
    'pontocerebellar fibers':           ('ambiguous', 'Related to MCP; could be MCP synonym or distinct'),
    'pontocerebellar tract':            ('ambiguous', 'Related to MCP; verify'),
    'tractus pontocerebellaris':        ('ambiguous', 'Latin form of pontocerebellar tract'),
    'fornix':                           ('ambiguous', 'Whole fornix system; also clean_label of a row — exclude from synonym'),
    'uncinate fasciculus':              ('ambiguous', 'Exact match to clean_label — would be self-referential'),
    'corticospinal tract':              ('ambiguous', 'Exact match to clean_label — self-referential'),
    'medial lemniscus':                 ('ambiguous', 'Exact match to clean_label — self-referential'),
    'optic radiation':                  ('ambiguous', 'Exact match to clean_label — self-referential'),
    'middle longitudinal fasciculus':   ('ambiguous', 'Exact match to clean_label — self-referential'),
    'middle cerebellar peduncle':       ('ambiguous', 'Exact match to clean_label — self-referential'),
    'inferior cerebellar peduncle':     ('ambiguous', 'Exact match to clean_label — self-referential'),
    'superior cerebellar peduncle':     ('ambiguous', 'Exact match to clean_label — self-referential'),
    'extreme capsule':                  ('ambiguous', 'Exact match to clean_label — self-referential'),
    'frontal aslant tract':             ('ambiguous', 'Exact match to clean_label — self-referential'),
    'vertical occipital fasciculus':    ('ambiguous', 'Exact match to clean_label — self-referential'),
    'fronto-pontine tract':             ('ambiguous', 'Exact match to clean_label — self-referential'),
    'inferior longitudinal fasciculus': ('ambiguous', 'Exact match to clean_label — self-referential'),
    'inferior fronto-occipital fasciculus': ('ambiguous', 'Exact match to clean_label — self-referential'),
    'superior longitudinal fasciculus': ('ambiguous', 'Exact match to clean_label — self-referential'),
    'anterior thalamic radiation':      ('ambiguous', 'Exact match to clean_label — self-referential'),
    'arcuate fasciculus':               ('ambiguous', 'Exact match to clean_label — self-referential'),
    'posterior arcuate fasciculus':     ('ambiguous', 'Exact match to clean_label — self-referential'),
    'corticopontine tract':             ('ambiguous', 'Exact match to clean_label — self-referential'),

    # ── NEW TRACTS: valid tracts not yet in master ─────────────────────────
    'dentatothalamic tract':            ('new_tract', 'dentato-rubro-thalamic tract'),
    'dentatothalamic fibers':           ('new_tract', 'dentato-rubro-thalamic tract'),
    'tractus dentatothalamicus':        ('new_tract', 'dentato-rubro-thalamic tract'),
}

# ── Tract-specific synonym map: synonym → target clean_label ──────────────
# (all entries not in CLASSIFICATION above are mapped here)
SYNONYM_TO_TRACT = {
    # Anterior commissure
    'anterior cerebral commissure':         'anterior commissure',
    'anterior commissural nucleus':         'anterior commissure',
    'anterior commissure':                  'anterior commissure',
    'commissura anterior':                  'anterior commissure',
    'commissura anterior cerebri':          'anterior commissure',
    'commissura rostral':                   'anterior commissure',
    'commissura rostralis':                 'anterior commissure',
    'paleocortical commissure':             'anterior commissure',
    'precommisure':                         'anterior commissure',
    'rostral commissure':                   'anterior commissure',
    'structure of anterior commissure':     'anterior commissure',

    # Anterior thalamic radiation
    'anterior radiation of thalamus':       'anterior thalamic radiation',
    'anterior thalamic radiations':         'anterior thalamic radiation',
    'radiatio thalami anterior':            'anterior thalamic radiation',
    'radiationes thalamicae anteriores':    'anterior thalamic radiation',

    # Arcuate fasciculus
    'arcuate fascicle':                     'arcuate fasciculus',
    'cerebral arcuate fasciculus':          'arcuate fasciculus',
    'fasciculus arcuatus':                  'arcuate fasciculus',
    'fibrae arcuatae cerebri':              'arcuate fasciculus',
    'posteior arcuate fascisculus':         'arcuate fasciculus',  # typo in source

    # Callosum forceps major
    'corpus callosum, forceps major':       'callosum forceps major',
    'corpus callosum, posterior forceps (arnold)': 'callosum forceps major',
    'forceps major':                        'callosum forceps major',
    'forceps major corporis callosi':       'callosum forceps major',
    'forceps major of corpus callosum':     'callosum forceps major',
    'forceps major of the corpus callosum': 'callosum forceps major',
    'forceps occipitalis':                  'callosum forceps major',
    'major forceps':                        'callosum forceps major',
    'occipital forceps':                    'callosum forceps major',
    'posterior forceps':                    'callosum forceps major',
    'posterior forceps of corpus callosum': 'callosum forceps major',
    'posterior forceps of the corpus callosum': 'callosum forceps major',
    'structure of forceps major':           'callosum forceps major',

    # Callosum forceps minor
    'anterior forceps':                     'callosum forceps minor',
    'anterior forceps of corpus callosum':  'callosum forceps minor',
    'anterior forceps of the corpus callosum': 'callosum forceps minor',
    'corpus callosum, anterior forceps':    'callosum forceps minor',
    'corpus callosum, anterior forceps (arnold)': 'callosum forceps minor',
    'corpus callosum, forceps minor':       'callosum forceps minor',
    'forceps frontalis':                    'callosum forceps minor',
    'forceps minor':                        'callosum forceps minor',
    'forceps minor corporis callosi':       'callosum forceps minor',
    'forceps minor of corpus callosum':     'callosum forceps minor',
    'forceps minor of the corpus callosum': 'callosum forceps minor',
    'frontal forceps':                      'callosum forceps minor',
    'minor forceps':                        'callosum forceps minor',
    'structure of forceps minor':           'callosum forceps minor',

    # Cingulum bundle dorsal
    '(anterior) cingulum bundle':           'cingulum bundle dorsal',
    'neuraxis cingulum':                    'cingulum bundle dorsal',
    'cingulum of telencephalon':            'cingulum bundle dorsal',
    'structure of cerebral cingulum':       'cingulum bundle dorsal',
    'structure of cingulum':               'cingulum bundle dorsal',

    # Cingulum bundle ventral
    'cingulum (ammon\'s horn)':             'cingulum bundle ventral',
    'cingulum (hippocampus)':               'cingulum bundle ventral',
    'cingulum bundle in hippocampus':       'cingulum bundle ventral',

    # Cingulum cingulate
    'cingulum of brain':                    'cingulum cingulate',  # narrowed from cingulum ambig

    # Cingulum hippocampus

    # Corpus callosum subdivisions
    'corpus callosum - anterior midbody':   'corpus callosum anterior midbody',
    'corpus callosum - genu':               'corpus callosum genu',
    'corpus callosum - isthmus':            'corpus callosum isthmus',
    'corpus callosum - posterior midbody':  'corpus callosum posterior midbody',
    'corpus callosum - rostral body':       'corpus callosum rostral body',
    'corpus callosum - rostrum':            'corpus callosum rostrum',
    'corpus callosum - splenium':           'corpus callosum splenium',
    'body of corpus callosum':              'corpus callosum body central',  # nearest whole-body row
    'body of the corpus callosum':          'corpus callosum body central',
    'corpus callosum body':                 'corpus callosum body central',
    'corpus callosum truncus':              'corpus callosum body central',
    'truncus corporis callosi':             'corpus callosum body central',
    'trunculus corporis callosi':           'corpus callosum body central',
    'truncus corpus callosi':               'corpus callosum body central',
    'trunk of corpus callosum':             'corpus callosum body central',

    # Corticospinal tract
    'corticospinal fibers':                 'corticospinal tract',
    'fasciculus cerebro-spinalis':          'corticospinal tract',
    'fasciculus pyramidalis':               'corticospinal tract',
    'fibrae corticospinales':               'corticospinal tract',
    'pyramid (willis)':                     'corticospinal tract',
    'pyramidal tract':                      'corticospinal tract',
    'tractus cortico-spinalis':             'corticospinal tract',
    'tractus corticospinalis':              'corticospinal tract',
    'tractus pyramidalis':                  'corticospinal tract',
    'structure of corticospinal tract':     'corticospinal tract',

    # Corticopontine tract
    'cortico-pontine fibers':               'corticopontine tract',
    'cortico-pontine fibers, pontine part': 'corticopontine tract',
    'corticopontine':                       'corticopontine tract',
    'corticopontine fibers':                'corticopontine tract',
    'corticopontine fibers of pons':        'corticopontine tract',
    'corticopontine fibres':                'corticopontine tract',
    'corticopontine tract of pons':         'corticopontine tract',
    'fibrae corticopontinae':               'corticopontine tract',
    'tractus corticopontinus':              'corticopontine tract',
    'structure of corticopontine tract of pons': 'corticopontine tract',

    # Extreme capsule
    'band of baillarger':                   'extreme capsule',
    'capsula extrema':                      'extreme capsule',
    'structure of extreme capsule':         'extreme capsule',

    # External capsule (no standalone row — map to extreme capsule)
    'brain external capsule':              'extreme capsule',
    'capsula externa':                     'extreme capsule',
    'external capsule':                    'extreme capsule',
    'external capsule of telencephalon':   'extreme capsule',
    'structure of external capsule':       'extreme capsule',

    # Fornix
    'brain fornix':                         'fornix',
    'cerebral fornix':                      'fornix',
    'forebrain fornix':                     'fornix',
    'fornix (column and body of fornix)':   'fornix',
    'fornix cerebri':                       'fornix',
    'fornix hippocampus':                   'fornix',
    'fornix of brain':                      'fornix',
    'fornix of neuraxis':                   'fornix',
    'hippocampus fornix':                   'fornix',
    'neuraxis fornix':                      'fornix',

    # Frontal aslant tract
    'aslant tract':                         'frontal aslant tract',  # narrowed

    # Fronto-pontine tract
    'frontotemporal fasciculus':            'fronto-pontine tract',  # synonym used in some toolboxes

    # Inferior fronto-occipital fasciculus / inferior occipito-frontal
    'external sagittal stratum':            'inferior fronto-occipital fasciculus',
    'fasciculus fronto-occipitalis inferior':'inferior fronto-occipital fasciculus',
    'fasciculus occipito-frontalis inferior':'inferior fronto-occipital fasciculus',
    'fasciculus occipitofrontalis inferior': 'inferior fronto-occipital fasciculus',
    'inferior occipitofrontal fasciculus':  'inferior fronto-occipital fasciculus',
    'structure of inferior fronto-occipital fasciculus': 'inferior fronto-occipital fasciculus',

    # Inferior longitudinal fasciculus
    'fasciculus longitudinalis inferior':   'inferior longitudinal fasciculus',
    'structure of inferior longitudinal fasciculus': 'inferior longitudinal fasciculus',
    'temporooccipital fasciculus':          'inferior longitudinal fasciculus',

    # Medial lemniscus
    'lemniscus medialis':                   'medial lemniscus',

    # Middle longitudinal fasciculus
    'middle longitudinal fasciculus connection to the angular gyrus':    'middle longitudinal fasciculus',
    'middle longitudinal fasciculus connection to the superior parietal lobe': 'middle longitudinal fasciculus',

    # Optic radiation
    'genicula-celcarine tract':             'optic radiation',
    'geniculo-calcarine tract':             'optic radiation',
    'geniculocalcarine tract':              'optic radiation',
    'geniculostriate pathway':              'optic radiation',
    'gratiolet\'s radiation':              'optic radiation',
    'meyer\'s loop':                       'optic radiation',
    'optic radiations':                    'optic radiation',
    'radiatio optica':                     'optic radiation',
    'structure of optic radiation':        'optic radiation',

    # Olfactory tract (no standalone row — but has UBERON synonym data on extreme capsule row)
    'olfactory peduncle':                  'extreme capsule',
    'olfactory stalk':                     'extreme capsule',
    'olfactory tract':                     'extreme capsule',
    'pedunclulus olfactorius':             'extreme capsule',
    'tractus olfactorium':                 'extreme capsule',
    'tractus olfactorius':                 'extreme capsule',

    # Posterior arcuate fasciculus
    'posteior arcuate fascisculus':        'posterior arcuate fasciculus',

    # SLF subdivisions
    'superior longitudinal fascicle':      'superior longitudinal fasciculus',
    'superior longitudinal fascicle i':    'superior longitudinal fasciculus i',
    'superior longitudinal fascicle ii':   'superior longitudinal fasciculus ii',
    'superior longitudinal fascicle iii':  'superior longitudinal fasciculus iii',
    'structure of superior longitudinal fasciculus': 'superior longitudinal fasciculus',

    # Superior fronto-occipital fasciculus (no standalone row — synonym for superior LON)
    'fasciculus occipitofrontalis superior': 'superior longitudinal fasciculus',
    'fasciculus subcallosus':              'superior longitudinal fasciculus',
    'structure of superior fronto-occipital fasciculus': 'superior longitudinal fasciculus',
    'subcallosal bundle':                  'superior longitudinal fasciculus',
    'superior fronto-occipital bundle':    'superior longitudinal fasciculus',
    'superior fronto-occipital fasciculus':'superior longitudinal fasciculus',
    'superior occipito-frontal fascicle':  'superior longitudinal fasciculus',
    'superior occipitofrontal fasciculus': 'superior longitudinal fasciculus',

    # Striato- tracts
    'striato-fronto-orbital':              'striato-fronto-orbital tract',
    'striato-occipital':                   'striato-occipital tract',
    'striato-parietal':                    'striato-parietal tract',
    'striato-postcentral':                 'striato-postcentral tract',
    'striato-precentral':                  'striato-precentral tract',
    'striato-prefrontal':                  'striato-prefrontal tract',
    'striato-premotor':                    'striato-premotor tract',

    # Thalamo- tracts
    'thalamo-postcentral':                 'thalamo-postcentral tract',
    'thalamo-precentral':                  'thalamo-precentral tract',
    'thalamo-prefrontal':                  'thalamo-prefrontal tract',
    'thalamo-premotor':                    'thalamo-premotor tract',
    'temporo-parietal connections to the superior parietal lobule': 'temporo-parietal connection to superior parietal lobule',

    # Uncinate fasciculus
    'cerebral uncinate fasciculus':        'uncinate fasciculus',
    'hook bundle of russell':              'uncinate fasciculus',
    'russell\'s fasciculus':              'uncinate fasciculus',
    'structure of uncinate fasciculus':    'uncinate fasciculus',
    'tractus uncinatus':                   'uncinate fasciculus',
    'tractus uncinatus (lewandowsky)':     'uncinate fasciculus',
    'uncinate bundle of russell':          'uncinate fasciculus',
    'uncinate fascicle (russell)':         'uncinate fasciculus',
    'uncinate fasciculus of cerebellum':   'uncinate fasciculus',
    'uncinate fasciculus of pons':         'uncinate fasciculus',
    'uncinate fasciculus of russell':      'uncinate fasciculus',
    'uncinate fasciculus of the pons':     'uncinate fasciculus',
    'uncinate fasciculus-2':               'uncinate fasciculus',
    'tractus cerebello-bulbaris':          'uncinate fasciculus',

    # Vertical occipital fasciculus
    'perpendicular fasciculus':            'vertical occipital fasciculus',
    'structure of vertical occipital fasciculus': 'vertical occipital fasciculus',

    # Peduncle of midbrain (maps to cingulum given original UBERON data; flag as misc)
    'peduncle of midbrain':               'corticopontine tract',
    'pedunculi cerebri':                  'corticopontine tract',
    'pedunculus cerebralis':              'corticopontine tract',
    'pedunculus cerebri':                 'corticopontine tract',

    # Parieto-occipital pontine
    'parieto-occipital pontine':          'parieto-occipital pontine tract',

    # Superior thalamic radiation
    'superior thalamic radiation':        'superior thalamic radiation',

    # Corpus callosum: structure-label synonyms that can attach to the whole-CC rows
    'structure of tapetum of corpus callosum': 'corpus callosum splenium',
}

print(f'Classification rules defined: {len(CLASSIFICATION)} discard/ambiguous/new_tract entries')
print(f'Synonym→tract mappings defined: {len(SYNONYM_TO_TRACT)} entries')

Classification rules defined: 126 discard/ambiguous/new_tract entries
Synonym→tract mappings defined: 186 entries


## 3  Classify all input synonyms

In [10]:
def classify(clean_str):
    """Returns (category, target_or_note) for a normalised synonym string."""
    key = clean_str.strip().lower()

    # 1. Explicit classification rules take priority
    if key in CLASSIFICATION:
        return CLASSIFICATION[key]

    # 2. Self-referential: matches a clean_label exactly
    if key in label_to_idx:
        return ('ambiguous', f'Exact match to clean_label "{key}" — self-referential')

    # 3. Explicit tract mapping
    if key in SYNONYM_TO_TRACT:
        target = SYNONYM_TO_TRACT[key]
        if target in label_to_idx:
            return ('mapped', target)
        else:
            return ('ambiguous', f'Target "{target}" not found in master')

    # 4. Fallback: flag as ambiguous for manual review
    return ('ambiguous', 'No rule matched — needs manual review')

raw['category'] = raw['clean'].apply(lambda s: classify(s)[0])
raw['target_or_note'] = raw['clean'].apply(lambda s: classify(s)[1])

summary = raw.groupby('category').size().reset_index(name='count')
print(summary.to_string(index=False))
raw[['clean','category','target_or_note']].head(20)

 category  count
ambiguous     93
  discard     33
   mapped    181
new_tract      3


,clean,category,target_or_note
0,(Anterior) Cingulum Bundle,mapped,cingulum bundle dorsal
1,Anteriofrontal Corpus Callosum,ambiguous,Variant of anterior CC — overlaps CC_1/CC_2/CC_3
2,Anterior Cerebral Commissure,mapped,anterior commissure
3,Anterior Commissural Nucleus,mapped,anterior commissure
4,Anterior Commissure,ambiguous,"Exact match to clean_label ""anterior commissur..."
5,Anterior Corpus Callosum,ambiguous,Covers CC_1 rostrum through CC_4 anterior midbody
6,Anterior Forceps,mapped,callosum forceps minor
7,Anterior Forceps Of Corpus Callosum,mapped,callosum forceps minor
8,Anterior Forceps Of The Corpus Callosum,mapped,callosum forceps minor
9,Anterior Radiation Of Thalamus,mapped,anterior thalamic radiation


## 4  Save misc.csv (discard + ambiguous + new_tract)

In [11]:
misc = raw[raw['category'].isin(['discard','ambiguous','new_tract'])].copy()
misc = misc.rename(columns={'raw':'original_input','clean':'normalised','target_or_note':'note'})
misc = misc[['category','original_input','normalised','note']].sort_values(['category','normalised'])

misc.to_csv(OUT_MISC, index=False)
print(f'misc.csv saved: {len(misc)} rows')
misc.groupby('category').size()

misc.csv saved: 129 rows


category
ambiguous    93
discard      33
new_tract     3
dtype: int64

## 5  Integrate mapped synonyms into master

In [13]:
mapped = raw[raw['category'] == 'mapped'].copy()
print(f'Synonyms to integrate: {len(mapped)}')

def clean_term(s):
    """Normalise a synonym for storage: strip, collapse whitespace, no trailing commas."""
    s = s.strip()
    s = re.sub(r'\s+', ' ', s)
    s = s.strip(',')
    return s

def add_synonym_to_field(existing_field: str, new_term: str) -> str:
    """
    Insert new_term into a ' ; '-delimited synonym field.
    Deduplicates case-insensitively. Preserves commas within terms.
    """
    new_term = clean_term(new_term)
    if not new_term:
        return existing_field
    existing_terms = [t.strip() for t in existing_field.split(' ; ') if t.strip()]
    existing_lower = {t.lower() for t in existing_terms}
    if new_term.lower() not in existing_lower:
        existing_terms.append(new_term)
    return ' ; '.join(existing_terms)

# Group new synonyms by target tract
additions = mapped.groupby('target_or_note')['clean'].apply(list).to_dict()

# Apply to master dataframe
integration_log = []
for target_label, new_syns in additions.items():
    idx_list = [i for i, row in df.iterrows()
                if row['clean_label'].strip().lower() == target_label.lower()]
    if not idx_list:
        integration_log.append({'tract': target_label, 'status': 'NOT FOUND IN MASTER', 'synonyms': new_syns})
        continue
    for idx in idx_list:
        for syn in new_syns:
            df.at[idx, 'synonyms'] = add_synonym_to_field(df.at[idx, 'synonyms'], syn)
    integration_log.append({'tract': target_label, 'status': 'OK', 'synonyms': new_syns})

log_df = pd.DataFrame(integration_log)
print(log_df[['tract','status']].to_string(index=False))

Synonyms to integrate: 181
                                                  tract status
                                    anterior commissure     OK
                            anterior thalamic radiation     OK
                                     arcuate fasciculus     OK
                                 callosum forceps major     OK
                                 callosum forceps minor     OK
                                 cingulum bundle dorsal     OK
                                cingulum bundle ventral     OK
                       corpus callosum anterior midbody     OK
                           corpus callosum body central     OK
                                   corpus callosum genu     OK
                                corpus callosum isthmus     OK
                      corpus callosum posterior midbody     OK
                           corpus callosum rostral body     OK
                                corpus callosum rostrum     OK
                            

## 6  Handle new_tract entries: append to master with to_merge? flag

In [14]:
new_tracts = raw[raw['category'] == 'new_tract'].copy()

# Group synonyms by their suggested parent tract
grouped_new = new_tracts.groupby('target_or_note')['clean'].apply(list).to_dict()

empty_row = {col: '' for col in df.columns}
new_rows = []

for parent_suggestion, syns in grouped_new.items():
    # Use the first synonym as the clean_label, rest go into synonyms
    primary = syns[0]
    rest = ' ; '.join(syns[1:]) if len(syns) > 1 else ''
    row = empty_row.copy()
    row['clean_label'] = primary
    row['synonyms'] = rest
    row['to_merge?'] = f'Review: possible synonym of "{parent_suggestion}"'
    row['source_scheme (toolbox)'] = 'master_LUT'
    row['laterality'] = 'bilateral'
    new_rows.append(row)
    print(f'New row: "{primary}" | suggested parent: "{parent_suggestion}" | extra syns: {rest}')

if new_rows:
    df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)
    print(f'\nMaster now has {len(df)} rows')

New row: "dentatothalamic tract" | suggested parent: "dentato-rubro-thalamic tract" | extra syns: dentatothalamic fibers ; tractus dentatothalamicus

Master now has 78 rows


## 7  Verification

In [15]:
# Spot-check a few rows to verify synonyms were added correctly
checks = [
    'anterior commissure', 'arcuate fasciculus', 'callosum forceps major',
    'corticospinal tract', 'uncinate fasciculus', 'optic radiation',
    'inferior fronto-occipital fasciculus', 'striato-fronto-orbital tract',
    'thalamo-prefrontal tract', 'corpus callosum body central'
]

check_df = df[df['clean_label'].str.strip().str.lower().isin(checks)][['clean_label','acronym','synonyms']]
pd.set_option('display.max_colwidth', 120)
check_df

,clean_label,acronym,synonyms
1,anterior commissure,AC ; ACOMM ; CA,anterior cerebral commissure ; commissura anterior cerebri ; commissura rostralis ; commissura anterior ; commissura...
3,arcuate fasciculus,AF ; AR,fasciculus arcuatus ; cerebral arcuate fasciculus ; Arcuate Fascicle ; Fibrae Arcuatae Cerebri
4,callosum forceps major,FMA ; FMAJ,"forceps major ; occipital forceps ; posterior forceps ; forceps major of corpus callosum ; Corpus Callosum, Forceps ..."
12,corpus callosum body central,CC-BODY-C ; CC_BODYC,body of corpus callosum ; corpus callosum body ; truncus corporis callosi ; body of the corpus callosum ; corpus cal...
24,corticospinal tract,CST,pyramidal tract ; tractus corticospinalis ; fibrae corticospinales ; fasciculus pyramidalis ; tractus pyramidalis ; ...
33,inferior fronto-occipital fasciculus,IFO ; IFOF,inferior occipitofrontal fasciculus ; External Sagittal Stratum ; Fasciculus Fronto-Occipitalis Inferior ; Fasciculu...
40,optic radiation,OR,geniculocalcarine tract ; radiatio optica ; Optic radiation (body structure) ; Genicula-Celcarine Tract ; Geniculo-C...
44,striato-fronto-orbital tract,ST_FO,Striato-Fronto-Orbital
62,thalamo-prefrontal tract,T_PREF,Thalamo-Prefrontal
64,uncinate fasciculus,UF ; UNC,uncinate fasciculus of forebrain ; cerebral uncinate fasciculus ; Uncinate fasciculus (body structure) ; Hook Bundle...


In [16]:
# Count synonyms per row to see coverage improvement
df['_syn_count'] = df['synonyms'].apply(
    lambda s: len([x for x in s.split(' ; ') if x.strip()]) if s.strip() else 0
)
print('Synonym count distribution:')
print(df['_syn_count'].describe())
print(f'\nRows with 0 synonyms: {(df["_syn_count"]==0).sum()}')
print(f'Rows with 5+ synonyms: {(df["_syn_count"]>=5).sum()}')
df = df.drop(columns=['_syn_count'])

Synonym count distribution:
count    78.000000
mean      2.858974
std       4.332674
min       0.000000
25%       0.000000
50%       1.000000
75%       3.000000
max      16.000000
Name: _syn_count, dtype: float64

Rows with 0 synonyms: 28
Rows with 5+ synonyms: 16


## 8  Save final master

In [21]:
df.to_csv(OUT_MASTER, index=False)
print(f'Saved: {OUT_MASTER}')
print(f'Rows: {len(df)} | Columns: {len(df.columns)}')
print(f'\nColumns:')
for c in df.columns:
    print(f'  {c}')

Saved: csvOutput/8April_master_v5.csv
Rows: 77 | Columns: 18

Columns:
  clean_label
  Id_snomed
  Id_uberon
  Label_snomed
  Label_uberon
  acronym
  synonyms
  Definition
  laterality
  to_merge?
  source_scheme (toolbox)
  pyafq_name
  pyafq_set
  Incoming: subClassOf
  Incoming: part of - uberon
  Annotation: database_cross_reference
  Synonyms_uberon
  IRI
